In [18]:
from itertools import combinations
import pandas as pd
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [19]:
pharmacy_transactions = [
    {"Paracetamol", "Vitamin C", "Cough Syrup"},
    {"Paracetamol", "Antibiotic", "Vitamin C"},
    {"Antibiotic", "Vitamin C", "Pain Reliever"},
    {"Paracetamol", "Vitamin C", "Pain Reliever"},
    {"Paracetamol", "Antibiotic", "Vitamin C"},
    {"Vitamin C", "Cough Syrup"},
]

In [20]:
MIN_SUPPORT_COUNT_Q1 = 3

In [21]:
library_transactions = [
    {"Data Structures", "Python Programming", "DBMS"},
    {"Data Structures", "Operating Systems"},
    {"Python Programming", "DBMS", "Machine Learning"},
    {"Data Structures", "Python Programming", "Machine Learning"},
    {"Data Structures", "Python Programming", "DBMS"},
    {"Python Programming", "DBMS"},
]

In [22]:
MIN_SUPPORT_COUNT_Q2 = 3

In [24]:
def get_support_count(itemset, transactions):
  count = 0
  for t in transactions:
    if itemset.issubset(t):
      count+=1
    return count

In [25]:
def generate_C1(transactions):
  items = set()
  for t in transactions:
    items.update(t)
  return[frozenset([item]) for item in sorted(items)]

In [26]:
def filter_by_min_support(candidates,transactions,min_support_count):
  freq = {}
  rows = []
  for c in candidates:
    sup = get_support_count(c, transactions)
    rows.append((set(c),sup))
    if sup >= min_support_count:
      freq[c] = sup
  return freq, rows

In [27]:
def generate_candidates(prev_freq_itemsets, k):
  prev_items = list(prev_freq_itemsets.keys())
  candidates = set()
  for i in range(len(prev_items)):
    for j in range(i+1, len(prev_items)):
      union = prev_items[i] | prev_items[j]
      if len(union) == k:
        candidates.add(union)
  pruned = []
  for cand in candidates:
    subsets = combinations(cand, k-1)
    if all(frozenset(s) in prev_freq_itemsets for s in subsets):
      pruned.append(cand)
  return pruned

In [28]:
def print_table(title,rows,min_support_count):
  print(f"\n{title}")
  print("-"*len(title))
  for items,sup in sorted(rows,key=lambda r: sorted(r[0])):
    status = "FREQUENT" if sup >= min_support_count else "pruned (infrequent)"
    print(f"{sorted(items)} -> support count = {sup} [{status}]")

In [29]:
def run_apriori(transactions, min_support_count, dataset_name):
    print("\n" + "=" * 70)
    print(f"APRIORI ALGORITHM - {dataset_name}")
    print(f"Minimum support count = {min_support_count}")
    print("=" * 70)

## Question 1: Q1 - Hospital Pharmacy

In [35]:
transactions = pharmacy_transactions
min_support_count = MIN_SUPPORT_COUNT_Q1
dataset_name = "Q1 - Hospital Pharmacy"
all_frequent = {}

print("=" * 70)
print(f"APRIORI ALGORITHM - {dataset_name}")
print(f"Minimum support count = {min_support_count}")
print("=" * 70)

APRIORI ALGORITHM - Q1 - Hospital Pharmacy
Minimum support count = 3


In [36]:
C1 = generate_C1(transactions)
L1, rows1 = filter_by_min_support(C1, transactions, min_support_count)
print_table("C1 (Candidate 1-itemsets) -> L1 after pruning", rows1, min_support_count)
all_frequent.update(L1)


C1 (Candidate 1-itemsets) -> L1 after pruning
---------------------------------------------
  ['Antibiotic']  -> support count = 3   [FREQUENT]
  ['Cough Syrup']  -> support count = 2   [pruned (infrequent)]
  ['Pain Reliever']  -> support count = 2   [pruned (infrequent)]
  ['Paracetamol']  -> support count = 4   [FREQUENT]
  ['Vitamin C']  -> support count = 6   [FREQUENT]


In [37]:
C2 = generate_candidates(L1, 2)
L2, rows2 = filter_by_min_support(C2, transactions, min_support_count)
print_table("C2 (Candidate 2-itemsets) -> L2 after pruning", rows2, min_support_count)
all_frequent.update(L2)


C2 (Candidate 2-itemsets) -> L2 after pruning
---------------------------------------------
  ['Antibiotic', 'Paracetamol']  -> support count = 2   [pruned (infrequent)]
  ['Antibiotic', 'Vitamin C']  -> support count = 3   [FREQUENT]
  ['Paracetamol', 'Vitamin C']  -> support count = 4   [FREQUENT]


In [38]:
C3 = generate_candidates(L2, 3)
if C3:
    L3, rows3 = filter_by_min_support(C3, transactions, min_support_count)
    print_table("C3 (Candidate 3-itemsets) -> L3 after pruning", rows3, min_support_count)
    all_frequent.update(L3)
else:
    print("\nNo candidate 3-itemsets could be generated "
          "(join/prune step produced an empty C3) - algorithm stops here.")


No candidate 3-itemsets could be generated (join/prune step produced an empty C3) - algorithm stops here.


In [39]:
print(f"\nFinal list of ALL frequent itemsets for {dataset_name}:")
for items, sup in sorted(all_frequent.items(), key=lambda r: (len(r[0]), sorted(r[0]))):
    print(f"  {sorted(items)}  -> support count = {sup}")


Final list of ALL frequent itemsets for Q1 - Hospital Pharmacy:
  ['Antibiotic']  -> support count = 3
  ['Paracetamol']  -> support count = 4
  ['Vitamin C']  -> support count = 6
  ['Antibiotic', 'Vitamin C']  -> support count = 3
  ['Paracetamol', 'Vitamin C']  -> support count = 4


In [40]:
n = len(transactions)
te = TransactionEncoder()
te_array = te.fit([list(t) for t in transactions]).transform([list(t) for t in transactions])
df = pd.DataFrame(te_array, columns=te.columns_)
freq_items = apriori(df, min_support=min_support_count / n, use_colnames=True)
freq_items["support_count"] = (freq_items["support"] * n).round().astype(int)
print(f"--- mlxtend cross-check for {dataset_name} ---")
print(freq_items[["itemsets", "support", "support_count"]]
      .sort_values(by="support_count", ascending=False)
      .to_string(index=False))

--- mlxtend cross-check for Q1 - Hospital Pharmacy ---
                itemsets  support  support_count
             (Vitamin C) 1.000000              6
(Paracetamol, Vitamin C) 0.666667              4
           (Paracetamol) 0.666667              4
            (Antibiotic) 0.500000              3
 (Antibiotic, Vitamin C) 0.500000              3


In [41]:
print("Interpretation: Vitamin C appears in every prescription, and combinations such as {Paracetamol, Vitamin C} and {Antibiotic, Vitamin C} meet the minimum support count, so these medicines should be stocked / kitted together.")

Interpretation: Vitamin C appears in every prescription, and combinations such as {Paracetamol, Vitamin C} and {Antibiotic, Vitamin C} meet the minimum support count, so these medicines should be stocked / kitted together.


## Question 2: Q2 - University Library

In [42]:
transactions = library_transactions
min_support_count = MIN_SUPPORT_COUNT_Q2
dataset_name = "Q2 - University Library"
all_frequent = {}

print("=" * 70)
print(f"APRIORI ALGORITHM - {dataset_name}")
print(f"Minimum support count = {min_support_count}")
print("=" * 70)

APRIORI ALGORITHM - Q2 - University Library
Minimum support count = 3


In [43]:
C1 = generate_C1(transactions)
L1, rows1 = filter_by_min_support(C1, transactions, min_support_count)
print_table("C1 (Candidate 1-itemsets) -> L1 after pruning", rows1, min_support_count)
all_frequent.update(L1)


C1 (Candidate 1-itemsets) -> L1 after pruning
---------------------------------------------
  ['DBMS']  -> support count = 4   [FREQUENT]
  ['Data Structures']  -> support count = 4   [FREQUENT]
  ['Machine Learning']  -> support count = 2   [pruned (infrequent)]
  ['Operating Systems']  -> support count = 1   [pruned (infrequent)]
  ['Python Programming']  -> support count = 5   [FREQUENT]


In [44]:
C2 = generate_candidates(L1, 2)
L2, rows2 = filter_by_min_support(C2, transactions, min_support_count)
print_table("C2 (Candidate 2-itemsets) -> L2 after pruning", rows2, min_support_count)
all_frequent.update(L2)


C2 (Candidate 2-itemsets) -> L2 after pruning
---------------------------------------------
  ['DBMS', 'Data Structures']  -> support count = 2   [pruned (infrequent)]
  ['DBMS', 'Python Programming']  -> support count = 4   [FREQUENT]
  ['Data Structures', 'Python Programming']  -> support count = 3   [FREQUENT]


In [45]:
C3 = generate_candidates(L2, 3)
if C3:
    L3, rows3 = filter_by_min_support(C3, transactions, min_support_count)
    print_table("C3 (Candidate 3-itemsets) -> L3 after pruning", rows3, min_support_count)
    all_frequent.update(L3)
else:
    print("\nNo candidate 3-itemsets could be generated "
          "(join/prune step produced an empty C3) - algorithm stops here.")


No candidate 3-itemsets could be generated (join/prune step produced an empty C3) - algorithm stops here.


In [46]:
print(f"\nFinal list of ALL frequent itemsets for {dataset_name}:")
for items, sup in sorted(all_frequent.items(), key=lambda r: (len(r[0]), sorted(r[0]))):
    print(f"  {sorted(items)}  -> support count = {sup}")


Final list of ALL frequent itemsets for Q2 - University Library:
  ['DBMS']  -> support count = 4
  ['Data Structures']  -> support count = 4
  ['Python Programming']  -> support count = 5
  ['DBMS', 'Python Programming']  -> support count = 4
  ['Data Structures', 'Python Programming']  -> support count = 3


In [47]:
n = len(transactions)
te = TransactionEncoder()
te_array = te.fit([list(t) for t in transactions]).transform([list(t) for t in transactions])
df = pd.DataFrame(te_array, columns=te.columns_)
freq_items = apriori(df, min_support=min_support_count / n, use_colnames=True)
freq_items["support_count"] = (freq_items["support"] * n).round().astype(int)
print(f"--- mlxtend cross-check for {dataset_name} ---")
print(freq_items[["itemsets", "support", "support_count"]].sort_values(by="support_count", ascending=False).to_string(index=False))

--- mlxtend cross-check for Q2 - University Library ---
                             itemsets  support  support_count
                 (Python Programming) 0.833333              5
                               (DBMS) 0.666667              4
                    (Data Structures) 0.666667              4
           (DBMS, Python Programming) 0.666667              4
(Python Programming, Data Structures) 0.500000              3


In [48]:
print("Interpretation: {Python Programming, DBMS} and {Data Structures, Python Programming} are frequent itemsets, so the library should shelve these books near each other and recommend them together to students.")

Interpretation: {Python Programming, DBMS} and {Data Structures, Python Programming} are frequent itemsets, so the library should shelve these books near each other and recommend them together to students.
